In [1]:
import json
import random
import base64
import os
import shutil
import subprocess
import dill
import nltk
import config
import itertools
import io

from utils.DataHandlers import LatexDataHandler
from utils.ModelClient import ModelClient
from pathlib import Path
from collections import Counter
from pdf2image import convert_from_bytes

In [2]:
EDIT_TYPE = config.EDIT_TYPE
DATA_FOLDER_PATH = config.DATA_FOLDER_PATH
TASK_ID = 'identification_and_localization'
INPUT_TYPE = 'images'
COMPILATION_TYPE = config.COMPILATION_TYPE

In [3]:
# SYSTEM_PROMPT = """# Identify
# You are a helpful peer-review assistant.

# # Task
# You will be provided with a scientific manuscript. Your task is to identify and locate any errors, weaknesses, or inconsistencies between the claims made in the text and the values presented in the tables.

# # Output Format
# Your output should be in valid JSON format, with the following schema:

# {
#     "inconsistencies": list[list[str]],
# }

# For each error, weakness or inconsistency that you identify, you should return:
# * a string identifying the table (e.g. "Table 3")
# * the sentence in the manuscript that contains the false or weak claim, extracted verbatim from the manuscript
# * a short explanation of what the error, weakness, or inconsistency is

# Ensure that you include all sentences that contain claims that are only weakly supported or contradicted by data in the tables.
# """

# MESSAGE_TEMPLATE = """Identify any inconsistencies between the text and tables in the provided manuscript."""



In [4]:
SYSTEM_PROMPT = """# Identify
You are a helpful peer-review assistant.

# Task
You will be provided with a scientific manuscript. Your task is to identify and locate any errors, weaknesses, or inconsistencies between the claims made in the text and the values presented in the tables.

# Requirements
* You should be as specific as possible; try to provide the specific sentences or tables in the manuscript that are relevant to the identified error or weakness.
* You should be as comprehensive as possible; identify all weaknesses or errors in the manuscript, even if addressing one weakness would make other weaknesses less severe.
"""

MESSAGE_TEMPLATE = """Identify any inconsistencies between the text and tables in the provided manuscript."""



In [5]:
def compile_latex(latex_path):
    cmd = [
        'latexmk',
        '-pdf',
        #"-cd",  # run from source dir (helps relative paths)
        #f"-outdir={str(build_dir)}",
        latex_path.stem,
    ]

    try:
        proc = subprocess.run(
            cmd,
            cwd=str(latex_path.parent),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            timeout=600,  # 10 minutes per paper
        )

        return True
    except subprocess.TimeoutExpired:
        return False
    except Exception as e:
        print(e)
        return True

def project_compiled_latex_iterator(src_dir, dst_dir):
    num_files = len(os.listdir(src_dir))
    #do not iterate directly: iterate in order of number of edits
    for edit_num in range(num_files):
        edited_latex = f'edit_{edit_num}.tex'
        shutil.copy(src_dir / edited_latex, dst_dir / edited_latex)
        success = compile_latex(dst_dir / edited_latex)

        #compilation failed
        if not success:
            project_responses.append('')
            res_dict = {'res': ''}
            res_fp.write(json.dumps(res_dict) + '\n')
            res_fp.flush()
            continue

        paper_path = dst_dir / (edited_latex[:-3] + 'pdf')
        with open(paper_path, 'rb') as fp:
            data = fp.read()

        file_data = base64.b64encode(data).decode("utf-8")

        yield (SYSTEM_PROMPT, MESSAGE_TEMPLATE, file_data)

def compile_all_latex(src_dir, dst_dir):
    # compile all latex files
    tex_files = [f for f in os.listdir(src_dir) if f[-4:] == '.tex']
    manuscript_file_paths = []
    #do not iterate directly: iterate in order of number of edits so responses are in same order
    for edited_latex in tex_files:
        shutil.copy(src_dir / edited_latex, dst_dir / edited_latex)
        success = compile_latex(dst_dir / edited_latex)

        if not success:
            # if EDIT_TYPE == 'SEQUENTIAL':
            #     #stop if there is a compilation error: otherwise later files will also have issues
            #     break
            # else:
            #     #otherwise just ignore: we assume they're independent so order doesn't matter
            continue

        paper_path = dst_dir / (edited_latex[:-3] + 'pdf')

        # compilation still failed
        if not os.path.isfile(paper_path):
            continue

        manuscript_file_paths.append(paper_path)

    return manuscript_file_paths

In [ ]:
from pdf2image import convert_from_bytes
from PIL import Image
import math

# from prismm-bench code in llms-eval: https://github.com/EvolvingLMMs-Lab/lmms-eval/tree/main/lmms_eval/tasks/prismm_bench
def concat_images(images, column_num=3):
    if not images:
        return None

    if column_num == 1:
        total_height = images[0].height * len(images)
        total_width = images[0].width
    else:
        rows = math.ceil(len(images) / column_num)
        total_height = images[0].height * rows
        total_width = images[0].width * column_num

    concatenated_image = Image.new("RGB", (total_width, total_height), "white")

    x_offset, y_offset = 0, 0
    for cnt, image in enumerate(images):
        concatenated_image.paste(image, (x_offset, y_offset))
        x_offset += image.width
        if (cnt + 1) % column_num == 0:
            y_offset += image.height
            x_offset = 0

    return concatenated_image

def get_pdf_bytes(doc):
    pdf_data = doc.get("pdf")
    if pdf_data is None:
        raise ValueError("PDF data not found in document")
    return pdf_data.stream.getvalue()

# modified from prismm-bench code in llms-eval: https://github.com/EvolvingLMMs-Lab/lmms-eval/tree/main/lmms_eval/tasks/prismm_bench
def paper_to_img_grid(paper_path):
    visuals = []
    # pdf_bytes = get_pdf_bytes(doc)
    
    with open(paper_path, 'rb') as pdf:
        pdf_bytes = pdf.read()

    if not pdf_bytes:
        return visuals

    try:
        images = convert_from_bytes(pdf_bytes, dpi=144, last_page=50)  # safeguard for very large documents
        max_images = 5
        column_num = 3

        if not images:
            return visuals

        pages_per_image = math.ceil(len(images) / max_images)

        for i in range(0, len(images), pages_per_image):
            batch_pages = images[i : i + pages_per_image]
            concatenated_image = concat_images(batch_pages, column_num)
            if concatenated_image:
                visuals.append(concatenated_image.convert("RGB"))
            if len(visuals) >= max_images:
                break
    except Exception as e:
        print(e)
        pass

    return visuals

def prepare_image_req_data(paper_path):
    visuals = paper_to_img_grid(paper_path)

    img_data = []
    for vis in visuals:
        img_byte_arr = io.BytesIO()
        vis.save(img_byte_arr, format='PNG')
        img_byte_arr = img_byte_arr.getvalue()
        base64_img = base64.b64encode(img_byte_arr).decode('utf-8')
        img_data.append(base64_img)

    req_data = {
        'system_prompt': SYSTEM_PROMPT,
        'message': MESSAGE_TEMPLATE,
        'img_data': img_data,
    }

    return req_data

def manuscript_iterator(file_paths):
    for paper_path in file_paths:
        meta_id = paper_path.stem[5:] #combined paper + error id

        if INPUT_TYPE == 'pdf':
            with open(paper_path, 'rb') as fp:
                data = fp.read()
            file_data = base64.b64encode(data).decode("utf-8")
            req_data = {
                'system_prompt': SYSTEM_PROMPT,
                'message': MESSAGE_TEMPLATE,
                'file_data': file_data,
            }
        elif INPUT_TYPE == 'images':
            # Handle image input type
            req_data = prepare_image_req_data(paper_path)
        elif INPUT_TYPE == 'images_and_latex':
            pass

        # file_data = base64.b64encode(data).decode("utf-8")
        # meta_id = paper_path.stem[5:] #combined paper + error id
        # req_data = {
        #     'system_prompt': SYSTEM_PROMPT,
        #     'message': MESSAGE_TEMPLATE,
        #     'file_data': file_data,
        # }
        yield (meta_id, req_data)

In [7]:
os.makedirs('workspace/dataset/', exist_ok=True)

DETECTION_CLIENTS = []

for dmi in config.DETECTION_MODELS_INFO:
    DETECTION_CLIENTS.append(ModelClient(*dmi))

In [8]:
if not config.HUMAN_ANNOTATIONS:
    for model_client in DETECTION_CLIENTS:
        print(model_client.model_prefix_name)
        cache_path = f'{config.OUTPUT_FOLDER_PATH}responses_{TASK_ID}_{INPUT_TYPE}_{model_client.model_prefix_name}_cache'
        responses = []
        for project_id in project_ids:
            print(project_id)

            src_dir = Path('workspace/edited_latex_files') / project_id
            dst_dir = Path('workspace/dataset/')
            if COMPILATION_TYPE == 'JUST_IN_TIME':
                shutil.copytree(DATA_FOLDER_PATH / project_id, 'workspace/dataset/', dirs_exist_ok=True)
                manuscript_file_paths = compile_all_latex(src_dir, dst_dir)

            tex_filenames = set([f[:-4] for f in os.listdir(src_dir) if f[-4:] == '.tex'])
            pdf_filenames = set([f[:-4] for f in os.listdir(dst_dir / project_id) if f[-4:] == '.pdf'])
            manuscript_filenames = tex_filenames.intersection(pdf_filenames)
            manuscript_file_paths = [dst_dir / project_id / (filename + '.pdf') for filename in manuscript_filenames]

            for meta_id, res in model_client.iterative_query(manuscript_iterator(manuscript_file_paths)):
                paper_id = meta_id.split('___')[0]
                error_id = meta_id.split('___')[1]
                
                res_dict = {
                    'model_id': model_client.model_prefix_name,
                    'task_id': TASK_ID,
                    'input_type': INPUT_TYPE,
                    'response_text': res
                }
                
                responses.append((paper_id, error_id, res_dict))

        #     shutil.rmtree('workspace/dataset/')
        
        with open(f'{config.OUTPUT_FOLDER_PATH}responses_{TASK_ID}_{INPUT_TYPE}_{model_client.model_prefix_name}.dill', 'wb') as fp:
            dill.dump(responses, fp)

In [10]:
if config.HUMAN_ANNOTATIONS:
    for model_client in DETECTION_CLIENTS:
        with open('./output/annotations.dill', 'rb') as fp:
            human_annotations = dill.load(fp)

            project_ids = []
            for project_id, error_id, annotation in human_annotations:
                if annotation['human_validation_passed']:
                    project_ids.append((project_id, error_id, annotation['edited_pdf_path']))

        manuscript_file_paths = [project_data[2] for project_data in project_ids]
        cache_path = f'{config.OUTPUT_FOLDER_PATH}responses_{TASK_ID}_{INPUT_TYPE}_{model_client.model_prefix_name}_cache'
        for meta_id, res in model_client.iterative_query(manuscript_iterator(manuscript_file_paths), cache_file_path=cache_path):
            paper_id = meta_id.split('___')[0]
            error_id = meta_id.split('___')[1]

            res_dict = {
                'model_id': model_client.model_prefix_name,
                'task_id': TASK_ID,
                'input_type': INPUT_TYPE,
                'response_text': res
            }

            for annotation in human_annotations:
                if annotation[0] == paper_id and annotation[1] == error_id:
                    if 'model_responses' in annotation[2]:
                        annotation[2]['model_responses'].append(res_dict)
                    else:
                        annotation[2]['model_responses'] = [res_dict]
                    break

        with open(f'{config.OUTPUT_FOLDER_PATH}responses_{TASK_ID}_{INPUT_TYPE}_{model_client.model_prefix_name}.dill', 'wb') as fp:
            dill.dump(human_annotations, fp)


Loading cached responses from output/responses_identification_and_localization_images_gpt_52_mini_cache.dill
{'effort': 'none'}
{'effort': 'none'}
{'effort': 'none'}
{'effort': 'none'}
{'effort': 'none'}
{'effort': 'none'}
{'effort': 'none'}
{'effort': 'none'}
{'effort': 'none'}


In [12]:
with open(f'{config.OUTPUT_FOLDER_PATH}responses_{TASK_ID}_{INPUT_TYPE}_{model_client.model_prefix_name}.dill', 'rb') as fp:
    responses = dill.load(fp)

In [19]:
for project_id, err_id, res in responses:
    print(res['human_validation_passed'], len(res['model_responses']) if 'model_responses' in res else 0)

True 1
True 1
True 1
True 1
True 1
True 1
False 0
True 1
True 1
True 1
